In [ ]:
import torch
from torchvision import transforms

PARTE 2: PREPARACIÓN DATASET

In [ ]:
# transformaciones para entrenamiento

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# transformaciones para validación y test
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

PARTE 3: CARGAR EL DATASET

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/DATA_AAA/evenData'

# Check if the directory exists
if not os.path.exists(ruta):
    print(f"Error: The directory '{ruta}' does not exist.")
    print("Please make sure you have mounted Google Drive and that the path to your 'fotos_AAA' folder is correct.")
    # Initialize files as an empty list to prevent further errors if the directory is not found
    files = []
else:
    files = os.listdir(ruta)
    print(f"Total imágenes: {len(files)}'")
    print('Primeros 10: ')
    for f in files[:10]:
        print(f)


In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader


class ImageFolder(datasets.ImageFolder):
  def __getitem__(self, index):
    try:
      return super(ImageFolder, self).__getitem__(index)
    except:
      return self.__getitem__(index + 1)

# Load pre-split datasets from subfolders
train_dataset = ImageFolder(f"{ruta}/train", transform=transform_train)
val_dataset = ImageFolder(f"{ruta}/val", transform=transform_val)
test_dataset = ImageFolder(f"{ruta}/test", transform=transform_val)

print(f"Total imágenes en train: {len(train_dataset)}")
print(f"Total imágenes en val: {len(val_dataset)}")
print(f"Total imágenes en test: {len(test_dataset)}")

classes = train_dataset.classes # Define 'classes' variable from one of the datasets
print(f"Clases: {classes}")



PARTE 4: CARGAR DATOS

In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle=False)

print(f"Batches en train: {len(train_loader)}")
print(f"Batches en val: {len(val_loader)}")
print(f"Batches en test: {len(test_loader)}")


PARTE 5: ARQUITECTURA CNN

In [ ]:
import torch.nn as nn

class BaselineCNN(nn.Module):
  def __init__(self, num_classes=6, dropout_rate=0.5):
    super(BaselineCNN, self).__init__()


    # extracción características

    # bloque 1 : 3 canales RGB --> 32 filtros

    self.bloque1 = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(dropout_rate)
    )

    # bloque 2 : 32 filtros --> 64

    self.bloque2 = nn.Sequential(
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(dropout_rate)
    )

    # bloque 3 : 64 filtros -> 128 filtros

    self.bloque3 = nn.Sequential(
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(dropout_rate)
    )


    # clasificación

    self.clasificador = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 28 * 28, 256),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x = self.bloque1(x)
    x = self.bloque2(x)
    x = self.bloque3(x)
    x = self.clasificador(x)
    return x

In [ ]:
modelo = BaselineCNN(num_classes=6, dropout_rate=0.5)
print(modelo)

In [ ]:
# vamos a probar que la red funciona con el tamaño de la imágenes

imagen_prueba = torch.randn(1, 3, 224, 224)
with torch.no_grad():
  salida = modelo(imagen_prueba)

print(imagen_prueba.shape)
print(salida.shape)
print(classes)

PARTE 6: ENTRENAMIENTO

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo.to(device)

criterio = nn.CrossEntropyLoss()
optimizer = optim.Adam(modelo.parameters(), lr=0.001)
epochs = 20

loss_train_history = [] # Changed to store history of training loss per epoch
acc_val_history = []    # Changed to store history of validation accuracy per epoch

for epoch in range(epochs):

  # ENTRENAMIENTO

  modelo.train()
  loss_total = 0

  for imagenes, etiquetas in train_loader:
    imagenes = imagenes.to(device)
    etiquetas = etiquetas.to(device)

    optimizer.zero_grad() # resetea gradientes
    salida = modelo(imagenes) # forward pass
    loss = criterio(salida, etiquetas) # calcula error
    loss.backward() # backward pass
    optimizer.step() # actualizar pesos

    loss_total += loss.item()

  loss_medio_epoch = loss_total / len(train_loader) # Calculate average loss for the current epoch


  # fase de validación

  modelo.eval()
  correctas = 0
  total = 0

  with torch.no_grad():
    for imagenes, etiquetas in val_loader:
      imagenes = imagenes.to(device)
      etiquetas = etiquetas.to(device)
      salida = modelo(imagenes)
      _, predicciones = torch.max(salida, 1)
      correctas += (predicciones == etiquetas).sum().item()
      total += etiquetas.size(0)

  acc_val_epoch = correctas / total # Calculate validation accuracy for the current epoch

  loss_train_history.append(loss_medio_epoch) # Append epoch loss to history
  acc_val_history.append(acc_val_epoch)     # Append epoch accuracy to history

  print(f"Epoch: {epoch+1}/{epochs}")
  print(f"Loss train: {loss_medio_epoch:.4f}")
  print(f"Acc val: {acc_val_epoch:.4f}") # Print epoch accuracy


### PARTE 7: Visualización de los Resultados

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots
plt.figure(figsize=(12, 5))

# Plot training loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(range(1, epochs + 1), loss_train_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss per Epoch')
plt.grid(True)
plt.legend()

# Plot validation accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(range(1, epochs + 1), acc_val_history, label='Validation Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy per Epoch')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()
